# Week 05 — Neural Network Surrogates

Retrain NN surrogates with the new W4 data points (each function now has +1 sample).

**Context for W5 (per W4 results):** NNs beat baseline on 6/8 functions in W4 but **never topped any leaderboard**. Tuned GPs and SVRs consistently outperformed them in raw RMSE. The real value of NN surrogates this week is the **per-dimension gradient analysis** at each current-best point — `torch.autograd` gives a directional signal that complements the ensemble convergence analysis. Even when an NN doesn't beat baseline, its gradient (or lack of one) tells us something about local sensitivity.

For each function:
1. 5-fold CV across `H ∈ {16, 32}` × variants `{plain, dropout, wd, ensemble}`
2. Pick best (H, variant) by CV RMSE
3. Compare against baseline `Y.std()` — only mark as 'usable model' if it beats baseline
4. Refit on all data, save to `../models/week_05/function_N_nn.pt`
5. Save per-dim gradient at current best point (for reflection + decision-framework input)

**F1 extra:** retrain sign classifier with the new W4 sample.


In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_05')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 14 pts, 2D, baseline RMSE = 0.0019
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.0029 ← BEST
    H= 16  dropout     CV RMSE = 0.0029
    H= 32  ensemble    CV RMSE = 0.0041
    H= 32  wd          CV RMSE = 0.0045
    H= 16  ensemble    CV RMSE = 0.0046
    H= 32  plain       CV RMSE = 0.0049
    H= 16  plain       CV RMSE = 0.0059
    H= 16  wd          CV RMSE = 0.0059
  → best: H=32, dropout, RMSE=0.0029 (-52.1% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.0010000000474974513, 0.0]



F2: 14 pts, 2D, baseline RMSE = 0.2297
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.2305 ← BEST
    H= 16  dropout     CV RMSE = 0.2429
    H= 32  wd          CV RMSE = 0.2979
    H= 32  plain       CV RMSE = 0.3052
    H= 32  ensemble    CV RMSE = 0.3260
    H= 16  ensemble    CV RMSE = 0.3307
    H= 16  wd          CV RMSE = 0.3565
    H= 16  plain       CV RMSE = 0.3738
  → best: H=32, dropout, RMSE=0.2305 (-0.4% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [-0.5479999780654907, 0.1889999955892563]



F3: 19 pts, 3D, baseline RMSE = 0.0774
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0816 ← BEST
    H= 16  ensemble    CV RMSE = 0.0862
    H= 32  wd          CV RMSE = 0.0870
    H= 32  plain       CV RMSE = 0.0905
    H= 16  dropout     CV RMSE = 0.0905
    H= 16  plain       CV RMSE = 0.0923
    H= 16  wd          CV RMSE = 0.0942
    H= 32  dropout     CV RMSE = 0.0970
  → best: H=32, ensemble, RMSE=0.0816 (-5.5% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [-0.24899999797344208, 0.8220000267028809, 0.609000027179718]



F4: 34 pts, 4D, baseline RMSE = 8.5975
  All configs (sorted):
    H= 32  plain       CV RMSE = 3.8387 ← BEST
    H= 16  ensemble    CV RMSE = 4.0143
    H= 32  wd          CV RMSE = 4.0279
    H= 32  ensemble    CV RMSE = 4.2717
    H= 32  dropout     CV RMSE = 4.5880
    H= 16  wd          CV RMSE = 4.8929
    H= 16  dropout     CV RMSE = 4.9103
    H= 16  plain       CV RMSE = 4.9138
  → best: H=32, plain, RMSE=3.8387 (+55.4% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-15.348999977111816, 14.692999839782715, -6.315000057220459, -0.9290000200271606]



...

F6: 24 pts, 5D, baseline RMSE = 0.5772
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.3433 ← BEST
    H= 16  ensemble    CV RMSE = 0.3435
    H= 16  plain       CV RMSE = 0.3744
    H= 16  wd          CV RMSE = 0.3752
    H= 32  wd          CV RMSE = 0.3996
    H= 32  plain       CV RMSE = 0.4030
    H= 32  dropout     CV RMSE = 0.4391
    H= 16  dropout     CV RMSE = 0.4509
  → best: H=32, ensemble, RMSE=0.3433 (+40.5% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.41100001335144043, -0.49300000071525574, -0.4959999918937683, -0.2770000100135803, -0.8799999952316284]



F7: 34 pts, 6D, baseline RMSE = 0.4447
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.3239 ← BEST
    H= 16  ensemble    CV RMSE = 0.3357
    H= 32  dropout     CV RMSE = 0.3470
    H= 32  wd          CV RMSE = 0.3516
    H= 32  plain       CV RMSE = 0.3566
    H= 16  dropout     CV RMSE = 0.3662
    H= 16  plain       CV RMSE = 0.3766
    H= 16  wd          CV RMSE = 0.3774
  → best: H=32, ensemble, RMSE=0.3239 (+27.2% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-2.6500000953674316, 0.4320000112056732, -1.1150000095367432, 0.8709999918937683, -1.1269999742507935, 0.2619999945163727]



F8: 44 pts, 8D, baseline RMSE = 1.0735
  All configs (sorted):
    H= 32  plain       CV RMSE = 0.3980 ← BEST
    H= 32  ensemble    CV RMSE = 0.3995
    H= 32  wd          CV RMSE = 0.4052
    H= 16  ensemble    CV RMSE = 0.4067
    H= 16  dropout     CV RMSE = 0.4151
    H= 32  dropout     CV RMSE = 0.4158
    H= 16  wd          CV RMSE = 0.4880
    H= 16  plain       CV RMSE = 0.4963
  → best: H=32, plain, RMSE=0.3980 (+62.9% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.0820000022649765, -0.21199999749660492, 0.5490000247955322, -0.20000000298023224, 0.9929999709129333, 0.3490000069141388, -0.44200000166893005, -0.3140000104904175]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 14 pts, 10 positive, 4 zero-or-negative (71% positive)


Sign classifier trained. LOO accuracy = 64.29%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   32  dropout         0.0029      0.0019    -52.1%       ✗
 2   32  dropout         0.2305      0.2297     -0.4%       ✗
 3   32  ensemble        0.0816      0.0774     -5.5%       ✗
 4   32  plain           3.8387      8.5975    +55.4%       ✓
 5   32  plain         121.3938    536.6763    +77.4%       ✓
 6   32  ensemble        0.3433      0.5772    +40.5%       ✓
 7   32  ensemble        0.3239      0.4447    +27.2%       ✓
 8   32  plain           0.3980      1.0735    +62.9%       ✓
